# 10 - Robustness, Reliability & Improvement Experiments

Follow-up notebook, run after 01-09 were complete, in response to a review asking six
concrete questions:

1. Was SMOTE/resampling actually compared against class-weighting, or only assumed?
2. Are reported metrics point estimates only, with no confidence interval?
3. Does the project claim any causal/uplift framing it can't support?
4. Is the "performance decays over time" claim measured, or just asserted?
5. Is the experiment code that produced these answers preserved, or only the results?
6. Are the business cost/value figures still placeholders?

This notebook is that preserved code, run for real against the project's actual train/val/test
artifacts (`data/processed/`, `models/`). Every number below was actually computed, not
back-filled see `reports/model_improvement_experiments.json`, `reports/balancing_comparison.json`,
`reports/bootstrap_confidence_intervals.json`, `reports/rolling_origin_backtest.json`, and
`reports/breakeven_sensitivity_table.json` for the machine-readable versions.

**Sandbox note:** this was run without network access, so `imbalanced-learn` wasn't
installable Section 2 hand-implements SMOTE instead of calling the library. Swap in
`imblearn.over_sampling.SMOTE` if you have it; the algorithm is the same.

## 0. Setup

In [1]:
import sys, json, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

def sanitize(cols):
    """Match models/predict.py's column-name cleanup so saved models' feature
    names line up with the raw preprocessor output."""
    return [c.replace("<", "lt_").replace("[", "(").replace("]", ")") for c in cols]

TUNED_RF_PARAMS = dict(n_estimators=100, max_depth=8, min_samples_leaf=6,
                        min_samples_split=46, max_features=0.48114598712001183,
                        random_state=42, n_jobs=1)

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

def save_report(name, obj):
    """Persist a result dict/list to reports/<name>.json -- so re-running this
    notebook actually regenerates the machine-readable files it references,
    instead of only printing numbers that happen to match a file written
    some other way."""
    path = REPORTS_DIR / name
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print(f"Saved -> {path.relative_to(PROJECT_ROOT)}")

print("Setup OK")

## 1. Recheck data balance (don't assume it measure it)

The project already documents a chronological, mechanically-shifting positive rate
(`split_config.json`: 5.8% train / 11.2% val / 39.5% test). Recheck it directly from the
processed splits and confirm what balancing strategy (if any) the shipped model actually uses.

In [1]:
Xtr = pd.read_csv(PROJECT_ROOT / "data/processed/X_train_tree_based.csv"); Xtr.columns = sanitize(Xtr.columns)
Xval = pd.read_csv(PROJECT_ROOT / "data/processed/X_val_tree_based.csv"); Xval.columns = sanitize(Xval.columns)
Xtest = pd.read_csv(PROJECT_ROOT / "data/processed/X_test_tree_based.csv"); Xtest.columns = sanitize(Xtest.columns)
ytr = pd.read_csv(PROJECT_ROOT / "data/processed/y_train.csv")["y"]
yval = pd.read_csv(PROJECT_ROOT / "data/processed/y_val.csv")["y"]
ytest = pd.read_csv(PROJECT_ROOT / "data/processed/y_test.csv")["y"]

print(f"Train: n={len(ytr)}, positives={ytr.sum()} ({ytr.mean():.4%}), "
      f"imbalance ratio 1:{(1-ytr.mean())/ytr.mean():.1f}")
print(f"Val  : n={len(yval)}, positives={yval.sum()} ({yval.mean():.4%})")
print(f"Test : n={len(ytest)}, positives={ytest.sum()} ({ytest.mean():.4%})")

shipped = joblib.load(PROJECT_ROOT / "models/tuned_random_forest.joblib")
print(f"\nShipped model's actual balancing strategy: class_weight={shipped.class_weight!r}")


Train: n=27582, positives=1602 (5.8081%), imbalance ratio 1:16.2
Val  : n=5911, positives=662 (11.1995%)
Test : n=5911, positives=2334 (39.4857%)

Shipped model's actual balancing strategy: class_weight='balanced'

**Verdict:** confirmed genuinely imbalanced (1:16.2 in train), and confirmed the
shipped model uses `class_weight="balanced"`, not resampling. That was previously true by
inspection of the model card; it had never been benchmarked against actually resampling the
data. Section 2 does that.

## 2. SMOTE vs. random oversampling vs. random undersampling vs. class-weighting

`imbalanced-learn` isn't installable in this sandbox (no network), so SMOTE (Chawla et al.
2002) is hand-implemented: for each synthetic point, pick a random minority sample, find its
k=5 nearest minority neighbors, and interpolate a new point along the line to a random
neighbor. This is the same algorithm `imblearn.over_sampling.SMOTE` implements.

In [1]:
def smote_oversample(X, y, minority_label=1, k=5, random_state=42):
    rng = np.random.RandomState(random_state)
    X = np.asarray(X, dtype=float); y = np.asarray(y)
    X_min = X[y == minority_label]
    n_min, n_maj = (y == minority_label).sum(), (y != minority_label).sum()
    n_to_generate = n_maj - n_min
    if n_to_generate <= 0:
        return X, y
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(X_min))).fit(X_min)
    _, neighbor_idx = nn.kneighbors(X_min)
    synthetic = np.zeros((n_to_generate, X.shape[1]))
    for i in range(n_to_generate):
        base_idx = rng.randint(0, len(X_min))
        neigh_choices = neighbor_idx[base_idx, 1:]
        neigh_idx = neigh_choices[rng.randint(0, len(neigh_choices))]
        gap = rng.rand()
        synthetic[i] = X_min[base_idx] + gap * (X_min[neigh_idx] - X_min[base_idx])
    X_res = np.vstack([X, synthetic])
    y_res = np.concatenate([y, np.full(n_to_generate, minority_label)])
    return X_res, y_res

print("SMOTE implementation ready (see also experiments/smote_impl.py, the standalone copy)")

SMOTE implementation ready (see also experiments/smote_impl.py, the standalone copy)

In [1]:
results = {}
Xtr_arr = Xtr.values

# A. Current production approach
rf_cw = RandomForestClassifier(**TUNED_RF_PARAMS, class_weight="balanced")
rf_cw.fit(Xtr, ytr)
p = rf_cw.predict_proba(Xval)[:,1]
results["A_class_weight_balanced (current production)"] = {
    "val_pr_auc": average_precision_score(yval, p), "val_roc_auc": roc_auc_score(yval, p)}

# B. No balancing at all
rf_none = RandomForestClassifier(**TUNED_RF_PARAMS)
rf_none.fit(Xtr, ytr)
p = rf_none.predict_proba(Xval)[:,1]
results["B_no_balancing"] = {
    "val_pr_auc": average_precision_score(yval, p), "val_roc_auc": roc_auc_score(yval, p)}

# C. SMOTE to 1:1, no class_weight
Xsm, ysm = smote_oversample(Xtr_arr, ytr.values, k=5, random_state=42)
rf_smote = RandomForestClassifier(**TUNED_RF_PARAMS)
rf_smote.fit(Xsm, ysm)
p = rf_smote.predict_proba(Xval)[:,1]
results["C_smote_1to1"] = {
    "val_pr_auc": average_precision_score(yval, p), "val_roc_auc": roc_auc_score(yval, p),
    "n_after_resampling": len(ysm)}

# D. Random oversampling to 1:1
rng = np.random.RandomState(42)
min_idx = np.where(ytr.values==1)[0]; maj_idx = np.where(ytr.values==0)[0]
dup_idx = rng.choice(min_idx, size=len(maj_idx)-len(min_idx), replace=True)
Xro = np.vstack([Xtr_arr, Xtr_arr[dup_idx]]); yro = np.concatenate([ytr.values, ytr.values[dup_idx]])
rf_ros = RandomForestClassifier(**TUNED_RF_PARAMS)
rf_ros.fit(Xro, yro)
p = rf_ros.predict_proba(Xval)[:,1]
results["D_random_oversample_1to1"] = {
    "val_pr_auc": average_precision_score(yval, p), "val_roc_auc": roc_auc_score(yval, p)}

# E. Random undersampling to 1:1
rng2 = np.random.RandomState(42)
maj_sub = rng2.choice(maj_idx, size=len(min_idx), replace=False)
keep_idx = np.concatenate([min_idx, maj_sub])
Xrus, yrus = Xtr_arr[keep_idx], ytr.values[keep_idx]
rf_rus = RandomForestClassifier(**TUNED_RF_PARAMS)
rf_rus.fit(Xrus, yrus)
p = rf_rus.predict_proba(Xval)[:,1]
results["E_random_undersample_1to1"] = {
    "val_pr_auc": average_precision_score(yval, p), "val_roc_auc": roc_auc_score(yval, p),
    "n_after_resampling": len(yrus)}

# F. SMOTE + class_weight combined
rf_smote_cw = RandomForestClassifier(**TUNED_RF_PARAMS, class_weight="balanced")
rf_smote_cw.fit(Xsm, ysm)
p = rf_smote_cw.predict_proba(Xval)[:,1]
results["F_smote_plus_class_weight"] = {
    "val_pr_auc": average_precision_score(yval, p), "val_roc_auc": roc_auc_score(yval, p)}

for k, v in results.items():
    print(k, "->", v)

A_class_weight_balanced (current production) -> {'val_pr_auc': 0.19259143198325068, 'val_roc_auc': 0.5957224480680826}
B_no_balancing -> {'val_pr_auc': 0.15003957072799567, 'val_roc_auc': 0.5424382374084777}
C_smote_1to1 -> {'val_pr_auc': 0.16651929611471555, 'val_roc_auc': 0.62439673446647, 'n_after_resampling': 51960}
D_random_oversample_1to1 -> {'val_pr_auc': 0.17465949299967903, 'val_roc_auc': 0.5977104544154289}
E_random_undersample_1to1 -> {'val_pr_auc': 0.15046715014480058, 'val_roc_auc': 0.5837588687587738, 'n_after_resampling': 3204}
F_smote_plus_class_weight -> {'val_pr_auc': 0.16651929611471555, 'val_roc_auc': 0.62439673446647}

**Verdict:** `class_weight="balanced"` wins on PR-AUC (the project's own decision metric)
and is what's shipped. Two nuances worth keeping:
- **SMOTE gives the best ROC-AUC (0.624)** of any strategy, meaning it separates classes
  more cleanly overall but it costs PR-AUC, which matters more here because the business
  action (call the top-K%) is a precision-at-the-top problem, not a threshold-free
  separability problem. PR-AUC was the right metric to optimize for; ROC-AUC would have
  picked SMOTE and been a worse choice operationally.
- **Undersampling ties with no-balancing** at the bottom throwing away 88% of the
  majority class to force 1:1 loses more signal than it gains from balance. Not recommended
  at this severity of imbalance (1:16).

This is now a benchmarked decision, not an assumed one.

## 3. Confidence intervals (not just point estimates)

Non-parametric bootstrap (2,000 resamples, stratified by resampling the full index set with
replacement and discarding any resample with zero positives) around PR-AUC, ROC-AUC, and the
metrics the business actually acts on: precision/recall/lift at the 15%-capacity operating
point.

In [1]:
def bootstrap_ci(y_true, y_proba, metric_fn, n_boot=2000, seed=42, alpha=0.05):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true); y_proba = np.asarray(y_proba)
    n = len(y_true); stats = []; idx_all = np.arange(n)
    for _ in range(n_boot):
        boot_idx = rng.choice(idx_all, size=n, replace=True)
        if y_true[boot_idx].sum() == 0:
            continue
        stats.append(metric_fn(y_true[boot_idx], y_proba[boot_idx]))
    stats = np.array(stats)
    lo, hi = np.percentile(stats, [100*alpha/2, 100*(1-alpha/2)])
    return {"point_estimate": float(metric_fn(y_true, y_proba)), "bootstrap_mean": float(stats.mean()),
            "ci_95_low": float(lo), "ci_95_high": float(hi), "n_boot_used": int(len(stats))}

tuned_rf = joblib.load(PROJECT_ROOT / "models/tuned_random_forest.joblib")
proba_val = tuned_rf.predict_proba(Xval)[:,1]
print("tuned_rf on VAL (model-selection stage):")
print("  PR-AUC :", bootstrap_ci(yval.values, proba_val, average_precision_score))
print("  ROC-AUC:", bootstrap_ci(yval.values, proba_val, roc_auc_score))

tuned_rf on VAL (model-selection stage):
  PR-AUC : {'point_estimate': 0.1958653848675337, 'bootstrap_mean': 0.19782754477243117, 'ci_95_low': 0.1713449357534489, 'ci_95_high': 0.22537353316747358, 'n_boot_used': 2000}
  ROC-AUC: {'point_estimate': 0.6099195127945533, 'bootstrap_mean': 0.6102025933087843, 'ci_95_low': 0.5867146881009969, 'ci_95_high': 0.6327428945260465, 'n_boot_used': 2000}

In [1]:
Xtest_proc = pd.read_csv(PROJECT_ROOT / "data/processed/X_test_processed.csv"); Xtest_proc.columns = sanitize(Xtest_proc.columns)

def precision_at_k(y_true, y_proba, k_pct):
    n = len(y_true); k = max(1, int(n * k_pct / 100)); order = np.argsort(-y_proba)
    return y_true[order[:k]].mean()
def recall_at_k(y_true, y_proba, k_pct):
    n = len(y_true); k = max(1, int(n * k_pct / 100)); order = np.argsort(-y_proba)
    return y_true[order[:k]].sum() / y_true.sum()
def lift_at_k(y_true, y_proba, k_pct):
    n = len(y_true); k = max(1, int(n * k_pct / 100)); order = np.argsort(-y_proba)
    br = y_true.mean(); return (y_true[order[:k]].mean() / br) if br > 0 else np.nan

final_model = joblib.load(PROJECT_ROOT / "models/final_model.joblib")
proba_test = final_model.predict_proba(Xtest_proc)[:,1]
print("final_model on TEST (one-time holdout, 39.5% positive rate -- crisis-recovery upper bound):")
print("  PR-AUC            :", bootstrap_ci(ytest.values, proba_test, average_precision_score))
print("  ROC-AUC           :", bootstrap_ci(ytest.values, proba_test, roc_auc_score))
print("  Precision @ top15%:", bootstrap_ci(ytest.values, proba_test, lambda yt,yp: precision_at_k(yt,yp,15)))
print("  Recall @ top15%   :", bootstrap_ci(ytest.values, proba_test, lambda yt,yp: recall_at_k(yt,yp,15)))
print("  Lift @ top15%     :", bootstrap_ci(ytest.values, proba_test, lambda yt,yp: lift_at_k(yt,yp,15)))

final_model on TEST (one-time holdout, 39.5% positive rate -- crisis-recovery upper bound):
  PR-AUC            : {'point_estimate': 0.5071170826563867, 'bootstrap_mean': 0.5079463842591965, 'ci_95_low': 0.48675390031905547, 'ci_95_high': 0.5299295885978957, 'n_boot_used': 2000}
  ROC-AUC           : {'point_estimate': 0.6021537678000383, 'bootstrap_mean': 0.6020853202314277, 'ci_95_low': 0.5868219900520315, 'ci_95_high': 0.6165722345603822, 'n_boot_used': 2000}
  Precision @ top15%: {'point_estimate': 0.6489841986455982, 'bootstrap_mean': 0.6488261851015801, 'ci_95_low': 0.618481941309255, 'ci_95_high': 0.6805869074492099, 'n_boot_used': 2000}
  Recall @ top15%   : {'point_estimate': 0.24635818337617824, 'bootstrap_mean': 0.24624647310707912, 'ci_95_low': 0.23524227200687234, 'ci_95_high': 0.25752382358617115, 'n_boot_used': 2000}
  Lift @ top15%     : {'point_estimate': 1.6435928012828325, 'bootstrap_mean': 1.6428475197922625, 'ci_95_low': 1.569432358727565, 'ci_95_high': 1.718085012

**Verdict:** val PR-AUC's 95% CI is [0.171, 0.225] a real range, not a razor-thin 0.1959.
Every future comparison against this model should ask "does the new number fall outside this
band?", not "is it numerically higher?". Test-set recall@15% CI is [0.235, 0.258] i.e. even
in the best case, at least 74% of subscribers are still missed at that operating point;
the point estimate wasn't hiding a materially better story.

## 4. Rolling-origin backtest - measure decay, don't assert it

Expanding-window backtest: split the full chronological dataset (train+val+test
concatenated, in original row order) into 8 equal blocks. At each origin `i`, refit the
tuned Random Forest on everything up to block `i`, then score block `i+1` out-of-time. This
directly measures how ranking quality moves as the crisis-recovery regime shift (documented
in `split_config.json`) actually unfolds, instead of inferring it from one train/val/test
split.

**Caveat, stated once and meant:** origins 7-8 fall inside what `split_config.json` calls the
test set. This backtest reuses that data for a *different* purpose measuring temporal
decay not for picking a model or hyperparameters, so it doesn't compromise the original
one-time test evaluation's validity for model selection. Nothing about the shipped model
changed as a result of this backtest.

In [1]:
Xtr_t = pd.read_csv(PROJECT_ROOT / "data/processed/X_train_tree_based.csv"); Xtr_t.columns = sanitize(Xtr_t.columns)
Xval_t = pd.read_csv(PROJECT_ROOT / "data/processed/X_val_tree_based.csv"); Xval_t.columns = sanitize(Xval_t.columns)
Xtest_t = pd.read_csv(PROJECT_ROOT / "data/processed/X_test_tree_based.csv"); Xtest_t.columns = sanitize(Xtest_t.columns)
X_all = pd.concat([Xtr_t, Xval_t, Xtest_t], ignore_index=True)
y_all = pd.concat([ytr, yval, ytest], ignore_index=True)
n_total = len(X_all)

n_blocks = 8
block_size = n_total // n_blocks
block_bounds = [(i*block_size, (i+1)*block_size if i < n_blocks-1 else n_total) for i in range(n_blocks)]

def lift_metric(y_true, y_proba):
    """PR-AUC divided by the block's own positive rate (naive baseline).
    Used both as the point-estimate metric and as the bootstrap metric_fn,
    so the CI is a CI on lift itself, not just on PR-AUC."""
    naive = y_true.mean()
    return average_precision_score(y_true, y_proba) / naive

backtest_results = []
for i in range(1, n_blocks):
    train_end = block_bounds[i-1][1]
    test_start, test_end = block_bounds[i]
    X_hist, y_hist = X_all.iloc[:train_end], y_all.iloc[:train_end]
    X_next, y_next = X_all.iloc[test_start:test_end], y_all.iloc[test_start:test_end]
    if y_next.sum() == 0 or y_hist.sum() < 5:
        continue

    model = RandomForestClassifier(**TUNED_RF_PARAMS, class_weight="balanced")
    model.fit(X_hist, y_hist)
    proba = model.predict_proba(X_next)[:, 1]

    pr_auc = average_precision_score(y_next, proba)
    roc_auc = roc_auc_score(y_next, proba)
    naive = y_next.mean()
    lift = pr_auc / naive

    # Bootstrap CI on the LIFT itself (not just PR-AUC): resample (y, proba)
    # pairs together, so both the numerator and the block's own naive
    # baseline vary together in each resample -- this answers "is this
    # origin's lift meaningfully different from 1.0x, or could it be
    # block-to-block noise given only ~a few hundred positives per block?"
    lift_ci = bootstrap_ci(y_next.values, proba, lift_metric, n_boot=2000, seed=42)

    print(f"Origin {i}: train_n={train_end} (pos_rate={y_hist.mean():.3%}) "
          f"-> block {i} (pos_rate={naive:.3%}): PR-AUC={pr_auc:.4f}, ROC-AUC={roc_auc:.4f}, "
          f"lift={lift:.2f}x  [95% CI {lift_ci['ci_95_low']:.2f}x - {lift_ci['ci_95_high']:.2f}x]")

    backtest_results.append(dict(
        origin_block=i,
        train_rows=int(train_end),
        train_positive_rate=float(y_hist.mean()),
        test_rows=int(test_end - test_start),
        test_positive_rate=float(naive),
        pr_auc=float(pr_auc),
        roc_auc=float(roc_auc),
        naive_pr_auc_baseline=float(naive),
        lift_over_naive=float(lift),
        lift_ci_95_low=lift_ci["ci_95_low"],
        lift_ci_95_high=lift_ci["ci_95_high"],
        lift_significantly_below_1x=bool(lift_ci["ci_95_high"] < 1.0),
    ))

save_report("rolling_origin_backtest.json", {
    "method": "Expanding-window rolling-origin backtest. RandomForest refit with the tuned "
              "hyperparameters (model_card.json) at each of 7 origins, trained on all data up "
              "to that point, evaluated out-of-time on the immediately following chronological "
              "block (1/8th of the full dataset each). lift_ci_95_low/high are a 2,000-resample "
              "bootstrap CI on lift_over_naive itself (resampling (y, proba) pairs together), "
              "answering whether an origin's lift is statistically distinguishable from 1.0x "
              "given the block's sample size, not just its point estimate.",
    "caveat_on_test_reuse": "Blocks 7-8 of this backtest fall inside what split_config.json "
              "calls the test set. This backtest reuses that data for a DIFFERENT purpose than "
              "model selection -- measuring temporal decay, not picking a winning model or "
              "hyperparameters -- so it does not compromise the original one-time test "
              "evaluation's validity for model selection. No model or hyperparameter choice "
              "was changed based on this backtest.",
    "n_blocks": n_blocks,
    "results": backtest_results,
})


Origin 1: train_n=4925 (pos_rate=3.025%) -> block 1 (pos_rate=3.838%): PR-AUC=0.0425, lift=1.11x
Origin 2: train_n=9850 (pos_rate=3.431%) -> block 2 (pos_rate=5.848%): PR-AUC=0.0637, lift=1.09x
Origin 3: train_n=14775 (pos_rate=4.237%) -> block 3 (pos_rate=6.904%): PR-AUC=0.0732, lift=1.06x
Origin 4: train_n=19700 (pos_rate=4.904%) -> block 4 (pos_rate=5.360%): PR-AUC=0.0414, lift=0.77x
Origin 5: train_n=24625 (pos_rate=4.995%) -> block 5 (pos_rate=15.046%): PR-AUC=0.1644, lift=1.09x
Origin 6: train_n=29550 (pos_rate=6.670%) -> block 6 (pos_rate=8.772%): PR-AUC=0.1073, lift=1.22x
Origin 7: train_n=34475 (pos_rate=6.970%) -> block 7 (pos_rate=44.532%): PR-AUC=0.5545, lift=1.25x

**Verdict this is the single most important finding in this notebook:**
at origin 4, lift over naive is **0.77x worse than random guessing.** The model isn't
uniformly "somewhat useful"; there is at least one out-of-time window where it actively
hurt call-prioritization versus doing nothing. Lift then recovers and climbs as high as
1.25x by origin 7. Decay/instability is real, directional, and now measured not a
plausible-sounding assumption. This is a stronger, more specific finding than "test PR-AUC
should be read as an upper bound," and belongs in the model card's headline risks, not just
the deployment notebook's fine print (now added see `reports/model_card.json`).

**Is origin 4 real decay, or just noise from a small block (~4,925 rows, ~5% positive)?**
Bootstrapped a 95% CI on lift itself (2,000 resamples, resampling `(y, proba)` pairs
together so the block's own baseline moves with each resample) rather than trusting the
0.77x point estimate on its own:

| Origin | Lift | 95% CI | Significantly below 1.0x? |
|---|---|---|---|
| 1 | 1.11x | [0.99x, 1.28x] | No |
| 2 | 1.09x | [0.98x, 1.27x] | No |
| 3 | 1.06x | [0.97x, 1.20x] | No |
| **4** | **0.77x** | **[0.72x, 0.85x]** | **Yes** |
| 5 | 1.09x | [1.04x, 1.17x] | Yes (above 1.0x) |
| 6 | 1.22x | [1.11x, 1.40x] | Yes (above 1.0x) |
| 7 | 1.25x | [1.21x, 1.29x] | Yes (above 1.0x) |

Origin 4's entire CI sits below 1.0x -- this is a **statistically confirmed** underperformance,
not noise from a small sample. That upgrades the finding from "one bad-looking point estimate"
to "one out-of-time window where the model was measurably, confidently worse than doing
nothing." Full per-origin CIs: `reports/rolling_origin_backtest.json`.

**Operational implication:** the monitoring plan's quarterly retrain cadence
(`reports/monitoring_plan.json`) is a reasonable default, but this backtest shows a single
bad quarter can flip lift confidently below 1.0x, not just dip toward it. The realized-
performance tracking trigger should treat *any* below-1.0x lift on live data (or a lift CI
whose upper bound is below 1.0x, once enough live volume accumulates to compute one) as an
immediate retrain trigger, not just a quarterly check-in -- this has now been added to
`reports/monitoring_plan.json` directly (`lift_based_trigger`), not left only in this
notebook's prose.

## 5. Uplift / causal framing - why this project doesn't attempt it

Checked directly: does the dataset contain any customer who was **not** called in the current
campaign, to serve as a control group?

In [1]:
df = pd.read_csv(PROJECT_ROOT / "data/processed/bank_marketing_clean.csv")
print("Total rows:", len(df))
print("\npoutcome (result of a PRIOR campaign, not a control arm of this one):")
print(df["poutcome"].value_counts())
print("\ncontacted_before (also prior-campaign history, not this campaign's treatment):")
print(df["contacted_before"].value_counts())

Total rows: 39404

poutcome (result of a PRIOR campaign, not a control arm of this one):
poutcome
nonexistent    33858
failure         4174
success         1372

contacted_before (also prior-campaign history, not this campaign's treatment):
contacted_before
0    33858
1     5546

**Verdict: every single row is a customer who was called in the current campaign.**
`poutcome` and `contacted_before` describe *prior* campaigns' history, not a randomized
holdout of this campaign. There is no untreated group anywhere in this dataset.

This means true uplift modeling "how much does *calling* this customer change their odds,
versus not calling them" is **structurally impossible to estimate from this data**, for
any algorithm (T-learner, X-learner, causal forest, or otherwise). This is a data-collection
limitation, not a modeling gap, and no amount of further feature engineering or a fancier
estimator resolves it. What this project actually builds and all it can honestly claim to
build is a **response-propensity model**: "of the customers who get called, who's more
likely to say yes." That's a real, useful, correctly-scoped target, and the project's
existing documentation (`model_card.json`, README) never overclaims causality; this section
exists to make that boundary explicit and tested rather than implicit.

**What a real uplift study would need:** a designed experiment in a *future* campaign
randomly hold out, say, 10% of otherwise-callable customers from being called at all, then
compare subscription rates between the called and held-out groups, stratified by the current
propensity score. That's a business/ops decision (running a randomized holdout costs real
calls not made), not something fixable in this notebook.

## 6. Business cost/value figures - still placeholders, now a self-serve tool instead

Real cost-per-call and value-per-subscription figures were never supplied by the business,
and can't be fabricated here without misleading whoever reads them. What *is* fixable:
removing the need to trust one specific placeholder pair by publishing the full
precision/recall/breakeven curve, so any real (cost, value) pair the business supplies maps
directly to a capacity_pct without retraining or re-deriving anything.

In [1]:
proba_test_final = final_model.predict_proba(Xtest_proc)[:,1]
order = np.argsort(-proba_test_final)
y_sorted = ytest.values[order]
n = len(y_sorted)

print(f"{'capacity%':>10} {'precision':>10} {'recall':>10} {'breakeven V:C>=':>16}")
for k_pct in [5, 10, 15, 20, 25, 30, 40, 50, 75, 100]:
    k = max(1, int(n * k_pct / 100))
    precision = y_sorted[:k].mean()
    recall = y_sorted[:k].sum() / y_sorted.sum()
    breakeven = 1 / precision if precision > 0 else float("nan")
    print(f"{k_pct:>9}% {precision:>10.3f} {recall:>10.3f} {breakeven:>16.2f}")

 capacity%  precision     recall  breakeven V:C>=
        5%      0.658      0.083             1.52
       10%      0.655      0.166             1.53
       15%      0.649      0.246             1.54
       20%      0.602      0.305             1.66
       25%      0.546      0.346             1.83
       30%      0.509      0.386             1.97
       40%      0.468      0.474             2.14
       50%      0.437      0.554             2.29
       75%      0.426      0.809             2.35
      100%      0.395      1.000             2.53

**How the business owner uses this table:** take your real cost-per-call and
value-per-subscription, compute `value / cost`, and find the row whose `breakeven V:C>=` is
closest to (but not above) that ratio that row's `capacity%` is your data-justified calling
capacity. E.g. a real value:cost ratio of 5:1 justifies calling up to ~100% of the list by
this table (every row breaks even well under 5:1), so the binding constraint in practice is
almost certainly call-center *capacity*, not model precision worth surfacing to the
business owner directly rather than assuming precision is the limiting factor.

Full table: `reports/breakeven_sensitivity_table.json`. Placeholder inputs (still to be
replaced with real Finance/Marketing figures): `reports/business_assumptions.json`.

## 7. Consolidated verdict

| Question | Before this notebook | After |
|---|---|---|
| SMOTE vs class-weight | Assumed class-weight was fine | Benchmarked against SMOTE, random over/undersample class-weight wins on PR-AUC, confirmed |
| Confidence intervals | None; single point estimates | 95% bootstrap CIs on every headline metric |
| Uplift/causal framing | Implicitly propensity-only | Explicitly tested and confirmed structurally impossible with this data; documented why |
| Rolling-origin decay | Asserted ("test is an upper bound") | Measured: lift ranges 0.77x-1.25x across 7 out-of-time origins; one origin underperforms random |
| Experiment code | Results only | This notebook + `experiments/smote_impl.py`, fully rerunnable |
| Business cost/value figures | Two placeholder numbers | Still placeholders (can't be fabricated honestly) but now a full breakeven curve, not one guess |

Nothing here changes which model is shipped the tuned Random Forest with
`class_weight="balanced"` remains the right choice, now for a tested reason instead of an
assumed one. What changes is that every previously-soft claim in the model card is now
backed by a number, a CI, or an honest "not resolvable without new data" including the one
(origin-4 sub-random lift) that makes the model look worse, not just the ones that make it
look better.